# AI Trading System Phase 2: Quant Models Evaluation
**Objective:** Evaluate default parameters for tree-based models and Logistic Regression for the Soft Voting mechanism of the Quant Agent.

In [1]:
!pip install vnstock ta catboost

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 1.4 MB/s eta 0:00:00
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=e2651fd94b57578d6e760376312f09b61033e1141a36b46860750d5db1df7237
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Successfully built ta


In [2]:
import time
import numpy as np
import pandas as pd
from vnstock import Vnstock
import vnstock
import ta
import warnings
warnings.filterwarnings('ignore')

# Models
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression

# Evaluation & Validation
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import precision_score, f1_score, average_precision_score
from sklearn.preprocessing import StandardScaler

## 1. Data Preparation
Fetch 5-minute OHLCV historical data for the `VN30F1M` derivative symbol over the last 6 months using `vnstock` with source `'VCI'`.

In [7]:
!pip install --upgrade --no-cache-dir git+https://github.com/rongardF/tvdatafeed.git

  Cloning https://github.com/rongardF/tvdatafeed.git to /tmp/pip-req-build-fwsaw3eg
  Running command git clone --filter=blob:none --quiet https://github.com/rongardF/tvdatafeed.git /tmp/pip-req-build-fwsaw3eg
  Resolved https://github.com/rongardF/tvdatafeed.git to commit e6f6aaa7de439ac6e454d9b26d2760ded8dc4923
  Preparing metadata (setup.py) ... done
  Created wheel for tvdatafeed: filename=tvdatafeed-2.1.0-py3-none-any.whl size=17532 sha256=c20a2b4cbc9ce0ca8725851614e9ada46cff2129a4ab33b4256dc7c88bfe567d
  Stored in directory: /tmp/pip-ephem-wheel-cache-qqzu6c5d/wheels/88/4b/d2/ce1c8432f33d4ab6d148d6ece31055eadadda0a5dd9fb76366
Successfully built tvdatafeed


In [11]:
import pandas as pd
from tvDatafeed import TvDatafeed, Interval

print("Đang khởi tạo kết nối với TradingView (tvdatafeed)...")
# Khởi tạo không cần login
tv = TvDatafeed()

print("Đang kéo dữ liệu VN30F1M (Nến 5 phút, 5000 nến gần nhất)...")
# Lấy 5000 nến 5 phút từ sàn HNX (TradingView map VN30F1M vào sàn này)
df = tv.get_hist(symbol='VN30F1!', exchange='HNX', interval=Interval.in_5_minute, n_bars=5000)

if df is not None and not df.empty:
    print(f"✅ Thành công! Đã kéo được {len(df)} dòng dữ liệu.")

    # Chuẩn hóa lại tên cột cho giống với format bài toán
    df = df.reset_index()
    df = df.rename(columns={'datetime': 'time'})

    # Hiển thị 5 dòng đầu tiên để kiểm tra
    display(df.head())
else:
    print("❌ Lỗi: Không kéo được dữ liệu.")

Đang khởi tạo kết nối với TradingView (tvdatafeed)...
Đang kéo dữ liệu VN30F1M (Nến 5 phút, 5000 nến gần nhất)...


ERROR:tvDatafeed.main:Connection timed out
ERROR:tvDatafeed.main:no data, please check the exchange and symbol


❌ Lỗi: Không kéo được dữ liệu.


In [12]:
# Fetch 6 months of data
end_date = pd.Timestamp.today().strftime('%Y-%m-%d')
start_date = (pd.Timestamp.today() - pd.DateOffset(months=6)).strftime('%Y-%m-%d')

print(f"Fetching data from {start_date} to {end_date}...")

stock = Vnstock().stock(symbol='VN30F1M', source='VCI')
df = stock.quote.history(start=start_date, end=end_date, interval='5m')

# Memory requirement check: use 'time' or 'tradingDate' column to parse valid datetime
if 'time' in df.columns:
    df['time'] = pd.to_datetime(df['time'])
    df.set_index('time', inplace=True)
elif 'tradingDate' in df.columns:
    df['tradingDate'] = pd.to_datetime(df['tradingDate'])
    df.set_index('tradingDate', inplace=True)

# Rename columns to standard lowercase
df.columns = [c.lower() for c in df.columns]
print(f"Data fetched: {df.shape}")
df.head()

2026-03-24 14:52:11 - vnstock.common.data - INFO - Not a stock. Company and finance data unavailable.
INFO:vnstock.common.data:Not a stock. Company and finance data unavailable.


Fetching data from 2025-09-24 to 2026-03-24...


RetryError: RetryError[<Future at 0x7b46761f6450 state=finished raised ConnectionError>]

In [13]:
import pandas as pd
import numpy as np
from datetime import datetime

print("Tạo dữ liệu giả lập (Synthetic Data) để bypass tường lửa Codespace...")

# Khởi tạo 5000 cây nến 5-phút
np.random.seed(42)
dates = pd.date_range(end=datetime.now(), periods=5000, freq='5min')

# Sinh giá Random Walk (bắt đầu từ mốc 1250 điểm của VN30F1M)
close_prices = 1250 + np.random.randn(5000).cumsum() * 1.5

df = pd.DataFrame({
    'time': dates,
    'open': close_prices + np.random.randn(5000) * 0.5,
    'high': close_prices + np.abs(np.random.randn(5000)) * 2,
    'low': close_prices - np.abs(np.random.randn(5000)) * 2,
    'close': close_prices,
    'volume': np.random.randint(500, 10000, 5000)
})

print(f"✅ Thành công! Đã tạo {len(df)} dòng dữ liệu OHLCV.")
display(df.head())

Tạo dữ liệu giả lập (Synthetic Data) để bypass tường lửa Codespace...
✅ Thành công! Đã tạo 5000 dòng dữ liệu OHLCV.


,time,open,high,low,close,volume
0,2026-03-07 06:17:26.721748,1250.533191,1252.102061,1250.458225,1250.745071,9796
1,2026-03-07 06:22:26.721748,1250.310968,1251.148674,1250.472363,1250.537675,1541
2,2026-03-07 06:27:26.721748,1250.611386,1252.703970,1251.380618,1251.509208,5612
3,2026-03-07 06:32:26.721748,1253.628707,1254.014588,1251.900029,1253.793752,1207
4,2026-03-07 06:37:26.721748,1253.808937,1255.836879,1251.948088,1253.442522,7958


## 2. Feature Engineering
Calculate technical indicators: MACD, RSI(14), VWAP, and Bollinger Bands. Then, drop NaN values.

In [14]:
# Calculate RSI(14)
df['rsi_14'] = ta.momentum.RSIIndicator(close=df['close'], window=14).rsi()

# Calculate MACD
macd = ta.trend.MACD(close=df['close'])
df['macd'] = macd.macd()
df['macd_signal'] = macd.macd_signal()
df['macd_diff'] = macd.macd_diff()

# Calculate Bollinger Bands
bb = ta.volatility.BollingerBands(close=df['close'], window=20, window_dev=2)
df['bb_mavg'] = bb.bollinger_mavg()
df['bb_high'] = bb.bollinger_hband()
df['bb_low'] = bb.bollinger_lband()

# Calculate VWAP
df['vwap'] = ta.volume.VolumeWeightedAveragePrice(
    high=df['high'], low=df['low'], close=df['close'], volume=df['volume']
).volume_weighted_average_price()

# Drop rows with NaN values resulting from indicator lookback periods
df.dropna(inplace=True)
print(f"Data shape after feature engineering and dropping NaNs: {df.shape}")

Data shape after feature engineering and dropping NaNs: (4967, 14)


## 3. Labeling (Target Variable Generation)
Define 3 classes based on the next 5-minute candle's price change:
- `LONG` (1): Price increases by more than 1.0 point.
- `SHORT` (-1): Price decreases by more than 1.0 point.
- `HOLD` (0): Price changes by 1.0 point or less in either direction.

In [15]:
THRESHOLD = 1.0

# Calculate difference to next candle's close
df['next_close'] = df['close'].shift(-1)
df['price_change'] = df['next_close'] - df['close']

# Drop the last row as we don't know its future
df.dropna(inplace=True)

def assign_label(change):
    if change > THRESHOLD:
        return 1
    elif change < -THRESHOLD:
        return -1
    else:
        return 0

df['target'] = df['price_change'].apply(assign_label)

print("Class distribution:")
print(df['target'].value_counts(normalize=True))

Class distribution:
target
 0    0.501208
 1    0.250302
-1    0.248490
Name: proportion, dtype: float64


## 4. Modeling & Evaluation
Train models using `TimeSeriesSplit` cross-validation with default parameters. Compare their precision, Macro F1-Score, PR-AUC, and Inference Time (ms).

In [16]:
# Prepare features (X) and target (y)
features = [
    'open', 'high', 'low', 'close', 'volume',
    'rsi_14', 'macd', 'macd_signal', 'macd_diff',
    'bb_mavg', 'bb_high', 'bb_low', 'vwap'
]
X = df[features]
y = df['target']

# Define models with default parameters
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees': ExtraTreesClassifier(random_state=42),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1),
    'XGBoost': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss'),
    'CatBoost': CatBoostClassifier(random_state=42, verbose=0)
}

# Transform target to [0, 1, 2] since XGBoost expects classes starting from 0
# Original: -1 (SHORT), 0 (HOLD), 1 (LONG)
# Mapped: 0 (SHORT), 1 (HOLD), 2 (LONG)
y_mapped = y.map({-1: 0, 0: 1, 1: 2})

tscv = TimeSeriesSplit(n_splits=5)
results = []

for name, model in models.items():
    print(f"Evaluating {name}...")

    precisions, f1s, praucs, inf_times = [], [], [], []

    for train_index, test_index in tscv.split(X):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y_mapped.iloc[train_index], y_mapped.iloc[test_index]

        # Scaling is needed for LogisticRegression
        if name == 'LogisticRegression':
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
        else:
            X_test_scaled = X_test.values

        model.fit(X_train, y_train)

        # Prediction
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)

        # In sklearn/xgboost, predict might return 2D array, let's flatten
        if y_pred.ndim > 1:
            y_pred = y_pred.flatten()

        # Metrics
        # average='macro' calculates metrics for each label, and finds their unweighted mean.
        # This does not take label imbalance into account.
        precisions.append(precision_score(y_test, y_pred, average='macro', zero_division=0))
        f1s.append(f1_score(y_test, y_pred, average='macro', zero_division=0))

        # Multi-class PR-AUC (One-vs-Rest)
        try:
            # For PR-AUC, we use average_precision_score with macro average across classes
            # Note: requires scikit-learn >= 0.24 and probabilities
            # Convert y_test to one-hot for PR-AUC
            y_test_dummies = pd.get_dummies(y_test).values

            # Make sure shape matches (sometimes a class might not be present in test set)
            if y_test_dummies.shape[1] == y_pred_proba.shape[1]:
                prauc = average_precision_score(y_test_dummies, y_pred_proba, average='macro')
                praucs.append(prauc)
        except Exception as e:
            pass # Skip if error occurs due to missing classes

        # Inference Time test (1 row)
        single_row = X_test_scaled[[0]]
        start_time = time.perf_counter()
        _ = model.predict(single_row)
        end_time = time.perf_counter()
        inf_times.append((end_time - start_time) * 1000) # Convert to ms

    results.append({
        'Model': name,
        'Precision (Macro)': np.mean(precisions),
        'Macro F1-Score': np.mean(f1s),
        'PR-AUC': np.mean(praucs) if praucs else np.nan,
        'Inference Time (ms)': np.mean(inf_times)
    })

# Display Results
results_df = pd.DataFrame(results).set_index('Model')
results_df.sort_values('Precision (Macro)', ascending=False, inplace=True)
results_df

Evaluating LogisticRegression...
Evaluating RandomForest...
Evaluating ExtraTrees...
Evaluating LightGBM...
Evaluating XGBoost...
Evaluating CatBoost...


,Precision (Macro),Macro F1-Score,PR-AUC,Inference Time (ms)
Model,,,,
RandomForest,0.371534,0.270103,0.338726,11.492525
ExtraTrees,0.370351,0.272048,0.336759,21.533387
LightGBM,0.327499,0.294695,0.329849,1.519190
XGBoost,0.313124,0.284590,0.329631,1.170220
CatBoost,0.309478,0.281075,0.332298,1.069723
LogisticRegression,0.182586,0.222516,0.342074,0.289004
